# Q1 — Reproducible Data Pipeline

Builds the unified-schema feature store described in `SPEC.md` from the raw
EB-NeRD demo, EB-NeRD small, and MIND-small files. Each step below unifies
one raw table into the shared schema, then asserts the invariants that make
it safe to build on top of (Q2-Q4) and free of the leakage patterns called
out in Q9.

Run top-to-bottom (or via `python build_pipeline.py`, which executes this
notebook end-to-end) to rebuild `data/processed/` from raw files.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
EBNERD_DEMO = ROOT / "ebnerd_demo"
EBNERD_SMALL = ROOT / "ebnerd_small"
MIND_TRAIN = ROOT / "MINDsmall_train" / "MINDsmall_train"
MIND_DEV = ROOT / "MINDsmall_dev" / "MINDsmall_dev"
DATA_OUT = ROOT / "data" / "processed"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

ROOT

WindowsPath('C:/Users/HP/cs4406m26-assignment1c1')

## EB-NeRD -> unified `articles`

Maps EB-NeRD's `subtitle` to the shared `abstract` column (its closest
analogue to MIND's abstract) and keeps `body`, which MIND never has.

In [2]:
def build_ebnerd_articles(raw: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return pd.DataFrame({
        "article_id": prefix + raw["article_id"].astype(str),
        "dataset": prefix.rstrip("_"),
        "title": raw["title"],
        "abstract": raw["subtitle"],
        "body": raw["body"],
        "category": raw["category_str"],
        "subcategory": raw["subcategory"].apply(lambda ids: str(ids[0]) if len(ids) else None),
        "published_time": raw["published_time"],
    })


ebnerd_articles_raw = pd.read_parquet(EBNERD_DEMO / "articles.parquet")
ebnerd_articles = build_ebnerd_articles(ebnerd_articles_raw, "ebnerd_")

ebnerd_small_articles_raw = pd.read_parquet(EBNERD_SMALL / "articles.parquet")
ebnerd_small_articles = build_ebnerd_articles(ebnerd_small_articles_raw, "ebnerd_small_")

print("ebnerd shape:", ebnerd_articles.shape)
print("ebnerd_small shape:", ebnerd_small_articles.shape)
ebnerd_articles.head(5)

ebnerd shape: (11777, 8)
ebnerd_small shape: (20738, 8)


,article_id,dataset,title,abstract,body,category,subcategory,published_time
0,ebnerd_3037230,ebnerd,Ishockey-spiller: Jeg troede jeg skulle dø,"ISHOCKEY: Ishockey-spilleren Sebastian Harts håber stadig, at karrieren kan ...",Ambitionerne om at komme til USA og spille ishockey har 21-årige Sebastian H...,sport,327,2003-08-28 08:55:00
1,ebnerd_3044020,ebnerd,Prins Harry tvunget til dna-test,"Hoffet tvang Prins Harry til at tage dna-test fordi man frygtede, at Dianas ...",Den britiske tabloidavis The Sun fortsætter med at lække historier fra den k...,underholdning,432,2005-06-29 08:47:00
2,ebnerd_3057622,ebnerd,Rådden kørsel på blå plader,Kan ikke straffes: Udenlandske diplomater i Danmark slipper gratis fra sprit...,Slingrende spritkørsel. Grove overtrædelser af fartbegrænsningerne. Bunker a...,nyheder,133,2005-10-10 07:20:00
3,ebnerd_3073151,ebnerd,Mærsk-arvinger i livsfare,FANGET I FLODBØLGEN: Skibsrederens oldebørn måtte have hjælp efter flugt fra...,To oldebørn af skibsreder Mærsk McKinney Møller var i yderste livsfare under...,nyheder,133,2005-01-04 06:59:00
4,ebnerd_3193383,ebnerd,Skød svigersøn gennem babydyne,44-årig kvinde tiltalt for drab på ekssvigersøn sidste år og for at have med...,En 44-årig mormor blev i dag fremstillet i et nævningeting i Aalborg tiltalt...,krimi,NaN,2003-09-15 15:30:00


In [3]:
def test_ebnerd_articles_unified_schema():
    expected_cols = {
        "article_id", "dataset", "title", "abstract", "body",
        "category", "subcategory", "published_time",
    }
    for articles, raw, prefix, dataset_name in [
        (ebnerd_articles, ebnerd_articles_raw, "ebnerd_", "ebnerd"),
        (ebnerd_small_articles, ebnerd_small_articles_raw, "ebnerd_small_", "ebnerd_small"),
    ]:
        assert expected_cols == set(articles.columns)
        assert len(articles) == len(raw)
        assert articles["article_id"].is_unique
        assert (articles["dataset"] == dataset_name).all()
        assert articles["article_id"].str.startswith(prefix).all()


test_ebnerd_articles_unified_schema()
print("ok: EB-NeRD (demo + small) unified articles schema checks passed")

ok: EB-NeRD (demo + small) unified articles schema checks passed


## EB-NeRD -> unified `behaviors`

Combines the provider's `train/` and `validation/` impression logs into one
table - our own temporal split (below) decides `train`/`val`/`test`, not the
provider's file layout.

In [4]:
def build_ebnerd_behaviors(raw: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return pd.DataFrame({
        "impression_id": prefix + raw["impression_id"].astype(str),
        "dataset": prefix.rstrip("_"),
        "user_id": prefix + raw["user_id"].astype(str),
        "impression_time": raw["impression_time"],
        "article_ids_inview": raw["article_ids_inview"].apply(lambda ids: [prefix + str(i) for i in ids]),
        "article_ids_clicked": raw["article_ids_clicked"].apply(lambda ids: [prefix + str(i) for i in ids]),
        "session_id": prefix + raw["session_id"].astype(str),
    })


ebnerd_behaviors_raw = pd.concat([
    pd.read_parquet(EBNERD_DEMO / "train" / "behaviors.parquet"),
    pd.read_parquet(EBNERD_DEMO / "validation" / "behaviors.parquet"),
], ignore_index=True)
ebnerd_behaviors = build_ebnerd_behaviors(ebnerd_behaviors_raw, "ebnerd_")

ebnerd_small_behaviors_raw = pd.concat([
    pd.read_parquet(EBNERD_SMALL / "train" / "behaviors.parquet"),
    pd.read_parquet(EBNERD_SMALL / "validation" / "behaviors.parquet"),
], ignore_index=True)
ebnerd_small_behaviors = build_ebnerd_behaviors(ebnerd_small_behaviors_raw, "ebnerd_small_")

print("ebnerd shape:", ebnerd_behaviors.shape)
print("ebnerd_small shape:", ebnerd_small_behaviors.shape)
ebnerd_behaviors.head(5)

ebnerd shape: (50080, 7)
ebnerd_small shape: (477534, 7)


,impression_id,dataset,user_id,impression_time,article_ids_inview,article_ids_clicked,session_id
0,ebnerd_48401,ebnerd,ebnerd_22779,2023-05-21 21:06:50,"[ebnerd_9774516, ebnerd_9771051, ebnerd_9770028, ebnerd_9775402, ebnerd_9774...",[ebnerd_9759966],ebnerd_21
1,ebnerd_152513,ebnerd,ebnerd_150224,2023-05-24 07:31:26,"[ebnerd_9778669, ebnerd_9778736, ebnerd_9778623, ebnerd_9089120, ebnerd_9778...",[ebnerd_9778661],ebnerd_298
2,ebnerd_155390,ebnerd,ebnerd_160892,2023-05-24 07:30:33,"[ebnerd_9778369, ebnerd_9777856, ebnerd_9778500, ebnerd_9778021, ebnerd_9778...",[ebnerd_9777856],ebnerd_401
3,ebnerd_214679,ebnerd,ebnerd_1001055,2023-05-23 05:25:40,"[ebnerd_9776715, ebnerd_9776406, ebnerd_9776566, ebnerd_9776071, ebnerd_9776...",[ebnerd_9776566],ebnerd_1357
4,ebnerd_214681,ebnerd,ebnerd_1001055,2023-05-23 05:31:54,"[ebnerd_9775202, ebnerd_9776855, ebnerd_9776688, ebnerd_9771995, ebnerd_9776...",[ebnerd_9776553],ebnerd_1358


In [5]:
def test_ebnerd_behaviors_unified_schema():
    expected_cols = {
        "impression_id", "dataset", "user_id", "impression_time",
        "article_ids_inview", "article_ids_clicked", "session_id",
    }
    for behaviors, raw, dataset_name in [
        (ebnerd_behaviors, ebnerd_behaviors_raw, "ebnerd"),
        (ebnerd_small_behaviors, ebnerd_small_behaviors_raw, "ebnerd_small"),
    ]:
        assert expected_cols == set(behaviors.columns)
        assert len(behaviors) == len(raw)
        assert behaviors["impression_id"].is_unique
        assert (behaviors["dataset"] == dataset_name).all()
        bad_rows = behaviors.apply(
            lambda r: not set(r["article_ids_clicked"]).issubset(set(r["article_ids_inview"])),
            axis=1,
        )
        assert not bad_rows.any(), f"{dataset_name}: found clicked articles absent from the inview candidate set"


test_ebnerd_behaviors_unified_schema()
print("ok: EB-NeRD (demo + small) unified behaviors schema checks passed")

ok: EB-NeRD (demo + small) unified behaviors schema checks passed


## EB-NeRD -> unified `history`

One row per user (deduplicated across the `train/`/`validation/` history
files, which describe the same fixed pre-collection-window clicks).

In [6]:
def build_ebnerd_history(raw: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return pd.DataFrame({
        "user_id": prefix + raw["user_id"].astype(str),
        "dataset": prefix.rstrip("_"),
        "article_id_sequence": raw["article_id_fixed"].apply(lambda ids: [prefix + str(i) for i in ids]),
        "timestamp_sequence": raw["impression_time_fixed"],
        "read_time_sequence": raw["read_time_fixed"],
        "scroll_percentage_sequence": raw["scroll_percentage_fixed"],
    })


ebnerd_history_raw = pd.concat([
    pd.read_parquet(EBNERD_DEMO / "train" / "history.parquet"),
    pd.read_parquet(EBNERD_DEMO / "validation" / "history.parquet"),
], ignore_index=True).drop_duplicates(subset="user_id", keep="first")
ebnerd_history = build_ebnerd_history(ebnerd_history_raw, "ebnerd_")

ebnerd_small_history_raw = pd.concat([
    pd.read_parquet(EBNERD_SMALL / "train" / "history.parquet"),
    pd.read_parquet(EBNERD_SMALL / "validation" / "history.parquet"),
], ignore_index=True).drop_duplicates(subset="user_id", keep="first")
ebnerd_small_history = build_ebnerd_history(ebnerd_small_history_raw, "ebnerd_small_")

print("ebnerd shape (one row per user):", ebnerd_history.shape)
print("ebnerd_small shape (one row per user):", ebnerd_small_history.shape)
ebnerd_history.head(5)

ebnerd shape (one row per user): (1935, 6)
ebnerd_small shape (one row per user): (18827, 6)


,user_id,dataset,article_id_sequence,timestamp_sequence,read_time_sequence,scroll_percentage_sequence
0,ebnerd_13538,ebnerd,"[ebnerd_9738663, ebnerd_9738569, ebnerd_9738663, ebnerd_9738490, ebnerd_9738...","[2023-04-27T10:17:43.000000, 2023-04-27T10:18:01.000000, 2023-04-27T10:18:13...","[17.0, 12.0, 4.0, 5.0, 4.0, 9.0, 5.0, 46.0, 11.0, 10.0, 4.0, 0.0, 14.0, 10.0...","[100.0, 35.0, 100.0, 24.0, 100.0, 23.0, 100.0, 100.0, 100.0, 26.0, 100.0, na..."
1,ebnerd_58608,ebnerd,"[ebnerd_9739362, ebnerd_9739179, ebnerd_9738567, ebnerd_9739344, ebnerd_9739...","[2023-04-27T18:48:09.000000, 2023-04-27T18:48:45.000000, 2023-04-27T18:49:19...","[2.0, 24.0, 72.0, 65.0, 11.0, 4.0, 101.0, 0.0, 699.0, 19.0, 70.0, 18.0, 77.0...","[37.0, 61.0, 100.0, 100.0, 55.0, 100.0, 100.0, nan, 61.0, 100.0, 100.0, 100...."
2,ebnerd_95507,ebnerd,"[ebnerd_9739035, ebnerd_9738646, ebnerd_9634967, ebnerd_9738902, ebnerd_9735...","[2023-04-27T15:20:28.000000, 2023-04-27T15:20:47.000000, 2023-04-27T15:21:16...","[18.0, 29.0, 51.0, 12.0, 10.0, 10.0, 13.0, 24.0, 8.0, 26.0, 11.0, 16.0, 67.0...","[60.0, 100.0, 100.0, 21.0, 29.0, 67.0, 49.0, 54.0, 25.0, 19.0, 32.0, 51.0, 1..."
3,ebnerd_106588,ebnerd,"[ebnerd_9738292, ebnerd_9738216, ebnerd_9737266, ebnerd_9737556, ebnerd_9737...","[2023-04-27T08:29:09.000000, 2023-04-27T08:29:26.000000, 2023-04-27T08:30:00...","[9.0, 15.0, 42.0, 9.0, 3.0, 58.0, 26.0, 214.0, 12.0, 94.0, 8.0, 32.0, 41.0, ...","[24.0, 57.0, 100.0, nan, nan, 100.0, 100.0, 73.0, 26.0, 30.0, nan, 100.0, 88..."
4,ebnerd_617963,ebnerd,"[ebnerd_9739035, ebnerd_9739088, ebnerd_9738902, ebnerd_9738968, ebnerd_9738...","[2023-04-27T14:42:25.000000, 2023-04-27T14:43:10.000000, 2023-04-27T14:43:39...","[45.0, 29.0, 116.0, 26.0, 34.0, 42.0, 58.0, 59.0, 65.0, 215.0, 113.0, 34.0, ...","[100.0, 100.0, nan, 46.0, 23.0, 19.0, 61.0, 70.0, 64.0, 72.0, 100.0, 29.0, n..."


In [7]:
def test_ebnerd_history_unified_schema():
    expected_cols = {
        "user_id", "dataset", "article_id_sequence",
        "timestamp_sequence", "read_time_sequence", "scroll_percentage_sequence",
    }
    for history, dataset_name in [(ebnerd_history, "ebnerd"), (ebnerd_small_history, "ebnerd_small")]:
        assert expected_cols == set(history.columns)
        assert history["user_id"].is_unique
        assert (history["dataset"] == dataset_name).all()
        row = history.iloc[0]
        lengths = {len(row[c]) for c in [
            "article_id_sequence", "timestamp_sequence", "read_time_sequence", "scroll_percentage_sequence",
        ]}
        assert len(lengths) == 1, f"{dataset_name}: the four parallel history arrays must have equal length per user"


test_ebnerd_history_unified_schema()
print("ok: EB-NeRD (demo + small) unified history schema checks passed")

ok: EB-NeRD (demo + small) unified history schema checks passed


## MIND -> unified `articles`

`body` is always null - MSN's licensing terms mean the full article text was
never distributed (see `README.md`); it's a genuine dataset limitation, not a
parsing gap.

In [8]:
MIND_NEWS_COLS = [
    "news_id", "category", "subcategory", "title", "abstract",
    "url", "title_entities", "abstract_entities",
]
mind_articles_raw = pd.concat([
    pd.read_csv(MIND_TRAIN / "news.tsv", sep="\t", header=None, names=MIND_NEWS_COLS),
    pd.read_csv(MIND_DEV / "news.tsv", sep="\t", header=None, names=MIND_NEWS_COLS),
], ignore_index=True).drop_duplicates(subset="news_id", keep="first")


def build_mind_articles(raw: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "article_id": "mind_" + raw["news_id"].astype(str),
        "dataset": "mind",
        "title": raw["title"],
        "abstract": raw["abstract"],
        "body": None,
        "category": raw["category"],
        "subcategory": raw["subcategory"],
        "published_time": pd.NaT,
    })


mind_articles = build_mind_articles(mind_articles_raw)
print("shape:", mind_articles.shape)
mind_articles.head(5)

shape: (65238, 8)


,article_id,dataset,title,abstract,body,category,subcategory,published_time
0,mind_N55528,mind,"The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By","Shop the notebooks, jackets, and more that the royals can't live without.",None,lifestyle,lifestyleroyals,NaT
1,mind_N19639,mind,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding you back and keeping you from sh...,None,health,weightloss,NaT
2,mind_N61837,mind,The Cost of Trump's Aid Freeze in the Trenches of Ukraine's War,Lt. Ivan Molchanets peeked over a parapet of sand bags at the front line of ...,None,news,newsworld,NaT
3,mind_N53526,mind,I Was An NBA Wife. Here's How It Affected My Mental Health.,"I felt like I was a fraud, and being an NBA wife didn't help that. In fact, ...",None,health,voices,NaT
4,mind_N38324,mind,"How to Get Rid of Skin Tags, According to a Dermatologist","They seem harmless, but there's a very good reason you shouldn't ignore them...",None,health,medical,NaT


In [9]:
def test_mind_articles_unified_schema():
    expected_cols = {
        "article_id", "dataset", "title", "abstract", "body",
        "category", "subcategory", "published_time",
    }
    assert expected_cols == set(mind_articles.columns)
    assert mind_articles["article_id"].is_unique
    assert (mind_articles["dataset"] == "mind").all()
    assert mind_articles["body"].isna().all(), "MIND must never have body text (licensing)"


test_mind_articles_unified_schema()
print("ok: MIND unified articles schema checks passed")

ok: MIND unified articles schema checks passed


## MIND -> unified `behaviors`

`impressions` packs candidates and labels into one string (`news_id-label`
tokens); we split it into the same `article_ids_inview` / `article_ids_clicked`
shape as EB-NeRD.

In [10]:
MIND_BEHAVIORS_COLS = ["impression_id", "user_id", "time", "history", "impressions"]
mind_behaviors_raw = pd.concat([
    pd.read_csv(MIND_TRAIN / "behaviors.tsv", sep="\t", header=None, names=MIND_BEHAVIORS_COLS).assign(source_split="train"),
    pd.read_csv(MIND_DEV / "behaviors.tsv", sep="\t", header=None, names=MIND_BEHAVIORS_COLS).assign(source_split="dev"),
], ignore_index=True)
# provider train/dev files both restart impression_id at 1 -> must namespace by
# source file too, not just dataset, or concatenating collides train with dev.


def split_impressions(s: str):
    tokens = s.split()
    candidates = [t.rsplit("-", 1)[0] for t in tokens]
    labels = [int(t.rsplit("-", 1)[1]) for t in tokens]
    return candidates, labels


def build_mind_behaviors(raw: pd.DataFrame) -> pd.DataFrame:
    cand_labels = raw["impressions"].map(split_impressions)
    return pd.DataFrame({
        "impression_id": "mind_" + raw["source_split"] + "_" + raw["impression_id"].astype(str),
        "dataset": "mind",
        "user_id": "mind_" + raw["user_id"].astype(str),
        "impression_time": pd.to_datetime(raw["time"], format="%m/%d/%Y %I:%M:%S %p"),
        "article_ids_inview": cand_labels.map(lambda cl: ["mind_" + c for c in cl[0]]),
        "article_ids_clicked": cand_labels.map(lambda cl: ["mind_" + c for c, l in zip(*cl) if l == 1]),
        "session_id": None,
    })


mind_behaviors = build_mind_behaviors(mind_behaviors_raw)
print("shape:", mind_behaviors.shape)
mind_behaviors.head(5)

shape: (230117, 7)


,impression_id,dataset,user_id,impression_time,article_ids_inview,article_ids_clicked,session_id
0,mind_train_1,mind,mind_U13740,2019-11-11 09:05:58,"[mind_N55689, mind_N35729]",[mind_N55689],None
1,mind_train_2,mind,mind_U91836,2019-11-12 18:11:30,"[mind_N20678, mind_N39317, mind_N58114, mind_N20495, mind_N42977, mind_N2240...",[mind_N17059],None
2,mind_train_3,mind,mind_U73700,2019-11-14 07:01:48,"[mind_N50014, mind_N23877, mind_N35389, mind_N49712, mind_N16844, mind_N5968...",[mind_N23814],None
3,mind_train_4,mind,mind_U34670,2019-11-11 05:28:05,"[mind_N35729, mind_N33632, mind_N49685, mind_N27581]",[mind_N49685],None
4,mind_train_5,mind,mind_U8125,2019-11-12 16:11:21,"[mind_N39985, mind_N36050, mind_N16096, mind_N8400, mind_N22407, mind_N60408...",[mind_N8400],None


In [11]:
def test_mind_behaviors_unified_schema():
    expected_cols = {
        "impression_id", "dataset", "user_id", "impression_time",
        "article_ids_inview", "article_ids_clicked", "session_id",
    }
    assert expected_cols == set(mind_behaviors.columns)
    assert len(mind_behaviors) == len(mind_behaviors_raw)
    assert mind_behaviors["impression_id"].is_unique
    assert (mind_behaviors["dataset"] == "mind").all()
    assert mind_behaviors["session_id"].isna().all(), "MIND has no session concept"
    bad_rows = mind_behaviors.apply(
        lambda r: not set(r["article_ids_clicked"]).issubset(set(r["article_ids_inview"])),
        axis=1,
    )
    assert not bad_rows.any(), "found clicked articles absent from the inview candidate set"


test_mind_behaviors_unified_schema()
print("ok: MIND unified behaviors schema checks passed")

ok: MIND unified behaviors schema checks passed


## MIND -> unified `history`

MIND has no dedicated history file - each `behaviors.tsv` row repeats the
same pre-log-window `history` string for a given user. We collapse that into
one row per user and **fail the build** if any user's `history` string isn't
identical across all their rows, since that would mean history was somehow
computed per-impression instead of being a fixed snapshot (the Q9 leakage
pattern this pipeline must never produce).

In [12]:
def build_mind_history(raw: pd.DataFrame) -> pd.DataFrame:
    non_null = raw.dropna(subset=["history"])
    distinct_counts = non_null.groupby("user_id")["history"].nunique()
    inconsistent = distinct_counts[distinct_counts > 1]
    if len(inconsistent):
        raise ValueError(
            f"{len(inconsistent)} MIND users have a history string that differs "
            "across their own impression rows - history must be a fixed "
            "pre-window snapshot, not something derived per-impression"
        )

    first_history = non_null.groupby("user_id")["history"].first()
    all_users = raw["user_id"].unique()
    sequences = {u: (first_history[u].split() if u in first_history.index else []) for u in all_users}

    return pd.DataFrame({
        "user_id": ["mind_" + u for u in all_users],
        "dataset": "mind",
        "article_id_sequence": [["mind_" + a for a in sequences[u]] for u in all_users],
        "timestamp_sequence": pd.array([None] * len(all_users), dtype="object"),
        "read_time_sequence": pd.array([None] * len(all_users), dtype="object"),
        "scroll_percentage_sequence": pd.array([None] * len(all_users), dtype="object"),
    })


mind_history = build_mind_history(mind_behaviors_raw)
print("shape (one row per user):", mind_history.shape)
mind_history.head(5)

shape (one row per user): (94057, 6)


,user_id,dataset,article_id_sequence,timestamp_sequence,read_time_sequence,scroll_percentage_sequence
0,mind_U13740,mind,"[mind_N55189, mind_N42782, mind_N34694, mind_N45794, mind_N18445, mind_N6330...",None,None,None
1,mind_U91836,mind,"[mind_N31739, mind_N6072, mind_N63045, mind_N23979, mind_N35656, mind_N43353...",None,None,None
2,mind_U73700,mind,"[mind_N10732, mind_N25792, mind_N7563, mind_N21087, mind_N41087, mind_N5445,...",None,None,None
3,mind_U34670,mind,"[mind_N45729, mind_N2203, mind_N871, mind_N53880, mind_N41375, mind_N43142, ...",None,None,None
4,mind_U8125,mind,"[mind_N10078, mind_N56514, mind_N14904, mind_N33740]",None,None,None


In [13]:
def test_mind_history_unified_schema():
    expected_cols = {
        "user_id", "dataset", "article_id_sequence",
        "timestamp_sequence", "read_time_sequence", "scroll_percentage_sequence",
    }
    assert expected_cols == set(mind_history.columns)
    assert mind_history["user_id"].is_unique
    assert (mind_history["dataset"] == "mind").all()
    # every user seen in behaviors gets a history row, even if empty (cold-start)
    assert set(mind_behaviors["user_id"]) == set(mind_history["user_id"])
    assert mind_history["timestamp_sequence"].isna().all(), "MIND never provides per-click timestamps"


test_mind_history_unified_schema()
print("ok: MIND unified history schema checks passed")

ok: MIND unified history schema checks passed


## Temporal split (train / val / test)

Both datasets ship only two provider splits (train, then dev/validation) -
there's no provider test set. We treat the provider's dev/validation split as
our held-out **test** set, and carve **val** from the last day of the
provider's train split by time. Cutoffs are real dates read off the raw
files (see `SPEC.md` section 3), not arbitrary constants.

In [14]:
EBNERD_TRAIN_END = pd.Timestamp("2023-05-24 07:00:00")   # last 24h of provider train -> val
EBNERD_TEST_START = pd.Timestamp("2023-05-25 07:00:00")  # == provider validation start
MIND_TRAIN_END = pd.Timestamp("2019-11-14 00:00:00")     # last day of provider train -> val
MIND_TEST_START = pd.Timestamp("2019-11-15 00:00:00")    # == provider dev start


def assign_split(times: pd.Series, train_end: pd.Timestamp, test_start: pd.Timestamp) -> pd.Series:
    return pd.Series(
        np.select([times < train_end, times < test_start], ["train", "val"], default="test"),
        index=times.index,
    )


# EB-NeRD small shares demo's exact provider date range (2023-05-18 -> 2023-06-01),
# so the same cutoffs apply -- verified directly on the raw files, not assumed.
ebnerd_behaviors["split"] = assign_split(ebnerd_behaviors["impression_time"], EBNERD_TRAIN_END, EBNERD_TEST_START)
ebnerd_small_behaviors["split"] = assign_split(ebnerd_small_behaviors["impression_time"], EBNERD_TRAIN_END, EBNERD_TEST_START)
mind_behaviors["split"] = assign_split(mind_behaviors["impression_time"], MIND_TRAIN_END, MIND_TEST_START)

ebnerd_behaviors["split"].value_counts(), ebnerd_small_behaviors["split"].value_counts(), mind_behaviors["split"].value_counts()

(split
 test     25356
 train    21318
 val       3406
 Name: count, dtype: int64,
 split
 test     244647
 train    200328
 val       32559
 Name: count, dtype: int64,
 split
 train    126695
 test      73152
 val       30270
 Name: count, dtype: int64)

In [15]:
def test_temporal_split_boundaries():
    for df, name in [(ebnerd_behaviors, "ebnerd"), (ebnerd_small_behaviors, "ebnerd_small"), (mind_behaviors, "mind")]:
        assert set(df["split"]) == {"train", "val", "test"}, f"{name}: missing a split"
        bounds = df.groupby("split")["impression_time"].agg(["min", "max"])
        assert bounds.loc["train", "max"] < bounds.loc["val", "min"], f"{name}: train/val overlap"
        assert bounds.loc["val", "max"] < bounds.loc["test", "min"], f"{name}: val/test overlap"


test_temporal_split_boundaries()
print("ok: temporal split boundaries are non-overlapping and monotonic (train < val < test)")

ok: temporal split boundaries are non-overlapping and monotonic (train < val < test)


## No future-click leakage (Q9)

Two checks, one per dataset, using whatever signal each actually provides:

- **EB-NeRD** has per-click timestamps in `history` - assert every historical
  click for a user happened strictly before that user's earliest logged
  impression.
- **MIND** has no history timestamps, so timestamp comparison isn't possible.
  The invariant that matters here already ran when `mind_history` was built
  (a `ValueError` would have aborted the build if any user's history string
  differed across impressions) - we assert it again explicitly, so the
  rebuild gate depends on a visible test, not just a constructor side effect.

In [16]:
def compute_ebnerd_leakage_check(history: pd.DataFrame, behaviors: pd.DataFrame) -> pd.DataFrame:
    hist_max = history.assign(
        max_ts=history["timestamp_sequence"].apply(lambda ts: max(ts) if len(ts) else pd.NaT)
    ).set_index("user_id")["max_ts"]
    beh_min = behaviors.groupby("user_id")["impression_time"].min()
    return pd.concat(
        [hist_max.rename("history_max_ts"), beh_min.rename("first_impression")], axis=1
    ).dropna()


ebnerd_leakage_check = compute_ebnerd_leakage_check(ebnerd_history, ebnerd_behaviors)
ebnerd_small_leakage_check = compute_ebnerd_leakage_check(ebnerd_small_history, ebnerd_small_behaviors)

mind_history_string_counts = mind_behaviors_raw.dropna(subset=["history"]).groupby("user_id")["history"].nunique()

ebnerd_leakage_check.head()

,history_max_ts,first_impression
user_id,,
ebnerd_13538,2023-05-17 20:36:34,2023-05-18 12:38:24
ebnerd_58608,2023-05-17 19:46:40,2023-05-18 14:49:45
ebnerd_95507,2023-05-17 14:57:46,2023-05-20 05:11:31
ebnerd_106588,2023-05-16 05:50:52,2023-05-18 14:11:26
ebnerd_617963,2023-05-18 02:28:09,2023-05-21 14:18:30


In [17]:
def test_no_future_click_leakage():
    for check, name in [(ebnerd_leakage_check, "EB-NeRD"), (ebnerd_small_leakage_check, "EB-NeRD small")]:
        violations = check[check["history_max_ts"] >= check["first_impression"]]
        assert len(violations) == 0, f"{name}: {len(violations)} users have history clicks at/after a logged impression"

    inconsistent = mind_history_string_counts[mind_history_string_counts > 1]
    assert len(inconsistent) == 0, f"MIND: {len(inconsistent)} users have a history string that varies across impressions"


test_no_future_click_leakage()
print("ok: no future-click leakage detected (Q9 behaviour-window boundary)")

ok: no future-click leakage detected (Q9 behaviour-window boundary)


## Write the feature store

One directory per dataset under `data/processed/` (gitignored - see Q8),
each with three parquet files plus a manifest recording split cutoffs, row
counts, and a build timestamp for reproducibility.

In [18]:
def write_feature_store(name: str, articles: pd.DataFrame, behaviors: pd.DataFrame, history: pd.DataFrame,
                         train_end: pd.Timestamp, test_start: pd.Timestamp) -> Path:
    out_dir = DATA_OUT / name
    out_dir.mkdir(parents=True, exist_ok=True)

    articles.to_parquet(out_dir / "articles.parquet", index=False)
    behaviors.to_parquet(out_dir / "behaviors.parquet", index=False)
    history.to_parquet(out_dir / "history.parquet", index=False)

    manifest = {
        "dataset": name,
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "split_cutoffs": {"train_end": str(train_end), "test_start": str(test_start)},
        "row_counts": {
            "articles": len(articles),
            "behaviors": len(behaviors),
            "history": len(history),
            "behaviors_by_split": behaviors["split"].value_counts().to_dict(),
        },
    }
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    return out_dir


ebnerd_out_dir = write_feature_store(
    "ebnerd", ebnerd_articles, ebnerd_behaviors, ebnerd_history, EBNERD_TRAIN_END, EBNERD_TEST_START,
)
ebnerd_small_out_dir = write_feature_store(
    "ebnerd_small", ebnerd_small_articles, ebnerd_small_behaviors, ebnerd_small_history,
    EBNERD_TRAIN_END, EBNERD_TEST_START,
)
mind_out_dir = write_feature_store(
    "mind", mind_articles, mind_behaviors, mind_history, MIND_TRAIN_END, MIND_TEST_START,
)
print("wrote:", ebnerd_out_dir)
print("wrote:", ebnerd_small_out_dir)
print("wrote:", mind_out_dir)

wrote: C:\Users\HP\cs4406m26-assignment1c1\data\processed\ebnerd
wrote: C:\Users\HP\cs4406m26-assignment1c1\data\processed\ebnerd_small
wrote: C:\Users\HP\cs4406m26-assignment1c1\data\processed\mind


In [19]:
def test_feature_store_roundtrip():
    for name, articles, behaviors, history, out_dir in [
        ("ebnerd", ebnerd_articles, ebnerd_behaviors, ebnerd_history, ebnerd_out_dir),
        ("ebnerd_small", ebnerd_small_articles, ebnerd_small_behaviors, ebnerd_small_history, ebnerd_small_out_dir),
        ("mind", mind_articles, mind_behaviors, mind_history, mind_out_dir),
    ]:
        for fname, expected in [
            ("articles.parquet", articles), ("behaviors.parquet", behaviors), ("history.parquet", history),
        ]:
            path = out_dir / fname
            assert path.exists(), f"{name}: missing {fname}"
            reloaded = pd.read_parquet(path)
            assert len(reloaded) == len(expected), f"{name}: {fname} row count mismatch after roundtrip"

        manifest_path = out_dir / "manifest.json"
        assert manifest_path.exists(), f"{name}: missing manifest.json"
        manifest = json.loads(manifest_path.read_text())
        assert manifest["row_counts"]["articles"] == len(articles)
        assert set(manifest["row_counts"]["behaviors_by_split"]) == {"train", "val", "test"}


test_feature_store_roundtrip()
print("ok: feature store round-trips correctly for all three datasets")

ok: feature store round-trips correctly for all three datasets


# Manual Review Complete